###  Notebook showing trained models being used to predict the test dataset split, creating the plots I'll show in the RESULTS section of readme

In [ ]:
import os
import torch
import logging
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from walkability.dataset.dataloader import get_data_loaders, LABEL_MAP
from walkability.train_model import train_model
from walkability.models.built_cnn import CNN
from walkability.models.finetuned_resnet import Pretrained_Resnet, Satellite_Resnet
from sklearn.metrics import accuracy_score


# dataset path
BASE_PATH = "/projects/dsci410_510/data/walkability_dataset/"
# select test data subset to make predictions
_, _, test_loader = get_data_loaders(BASE_PATH, name="all", batch_size=32)

INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()} # output labels

In [ ]:
# TODO: edit:
scratch_model = CNN.load_from_checkpoint("checkpoints/scratch/best.ckpt")
resnet_model  = Pretrained_Resnet.load_from_checkpoint("checkpoints/resnet/best.ckpt")

scratch_model.eval()
resnet_model.eval()

In [ ]:
# FUNC TO RUN PREDICTIONS ON TEST SET:
def get_predictions(model, loader):
    all_preds, all_labels, all_images = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            # predict
            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds)
            all_labels.append(labels)
            all_images.append(images)
    return (torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy(), torch.cat(all_images)) # preds, labels, data

scratch_preds, true_labels, test_images = get_predictions(scratch_model, test_loader)
resnet_preds, _, _ = get_predictions(resnet_model, test_loader)
# TODO: satellite resnet preds

In [ ]:
# Print accuracy summary table:

scratch_acc = accuracy_score(true_labels, scratch_preds)
resnet_acc  = accuracy_score(true_labels, resnet_preds)

print(f"{'Model':<20} {'Test Accuracy':>15}")
print("-" * 36)
print(f"{'Scratch CNN':<20} {scratch_acc:>15.4f}")
print(f"{'ResNet18':<20} {resnet_acc:>15.4f}")

In [ ]:
# confusion matrices

class_names = ["low", "medium", "high"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, preds, title in zip(axes,
                             [scratch_preds, resnet_preds],
                             ["Scratch CNN", "ResNet18"]):
    cm = confusion_matrix(true_labels, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{title} — Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()